In [3]:
import faiss
import numpy as np
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

from pathlib import Path
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv
from langchain_core.prompts import (
                                        SystemMessagePromptTemplate,
                                        HumanMessagePromptTemplate,
                                        PromptTemplate,
                                        ChatPromptTemplate
                                        )

env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)  # Load environment variables from workspace root .env file


True

In [2]:
llm_model_name = "qwen/qwen3-32b"

llm = ChatGroq(
    model=llm_model_name,
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
)


In [9]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

In [30]:
class company(BaseModel):
    name: str = Field(description="name of the company")
    year: int = Field(description="Revenue of the company in that year")

pydantic_parser = PydanticOutputParser(pydantic_object=company)

In [31]:
format_instructions=pydantic_parser.get_format_instructions()
format_instructions

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"name": {"description": "name of the company", "title": "Name", "type": "string"}, "year": {"description": "Revenue of the company in that year", "title": "Year", "type": "integer"}}, "required": ["name", "year"]}\n```'

In [32]:
query =  "what was the revenue of mfst in 2000?"

template = ChatPromptTemplate.from_template("""
Given the Query {query}, and the following retrieved documents, filter out the company name , year from the query'.
follow the {format_instructions}                                           
""")

chain = template | llm | pydantic_parser
chain_result = chain.invoke({"query": query , "format_instructions": format_instructions})

In [33]:
chain_result.name , chain_result.year

('MFST', 2000)